# Run review

Compare saved training runs and evaluation reports, export figures, and find recorded races for review.
Includes synchronized clip playback and editable individual/combined tactic annotations. Annotations are saved separately from recordings.

**Setup:** select the repository's `venv/bin/python` kernel in your IDE. For a fresh environment, run
`venv/bin/python -m pip install -e ".[analysis]"` from the repository root.
Launch in a browser with `venv/bin/python -m jupyter lab notebooks/run_review.ipynb`.
Run the cells in order, then edit the selection cell to choose runs. No training or model loading occurs.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "run.py").exists() and (p / "src" / "analysis").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the F110_MARL repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display
from src.analysis.run_review import (
    discover_runs, load_run, combine, latest_evaluations, summarize_races,
    summarize_agents, aggregate_training_seeds, filter_clips, replay_command, summarize_selfplay_evaluations,
)
from src.analysis.plots import (
    learning_curves, optimizer_curves, selfplay_curves, selfplay_evaluation_curves, evaluation_curves, outcome_plots,
    finish_time_plots, seed_variation_plots, export_figures,
)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
OUTPUT_ROOT = ROOT / "outputs"


## Available runs
Discovery reads small metadata/log files, never checkpoints or trajectory chunks.
Folders are distinct identities even when their saved `run_id` is the same.
Older runs without P0 race records remain visible, but cannot supply the new race-outcome plots.


In [ ]:
run_paths = discover_runs(OUTPUT_ROOT)
# Load only selected runs below. The inventory here reads metadata and small availability checks.
from src.analysis.run_review import read_json
inventory = []
for path in run_paths:
    snap = read_json(path / "config_snapshot.json")
    report = read_json(path / "evaluation_report.json")
    provenance = snap.get("provenance") or report.get("checkpoint_provenance") or {}
    inventory.append({
        "run": str(path.relative_to(OUTPUT_ROOT)), "scenario": provenance.get("scenario_name"),
        "training_seed": provenance.get("seed"),
        "training_races": (path / "race_metrics.jsonl").exists() or (path / "team_metrics.jsonl").exists(),
        "evaluation": (path / "evaluation_report.json").exists() or (path / "evaluation_history.jsonl").exists() or (path / "evaluation_races.jsonl").exists(),
        "recorded_clips": (path / "behavior" / "clips.jsonl").exists() or (path / "evaluation_behavior" / "clips.jsonl").exists(),
    })
display(pd.DataFrame(inventory))


## Select runs
Set `RUN_NAMES` to folder names from the inventory. With no selection, a single recent P0 run is opened
as a preview; select matched runs explicitly for comparisons. Nested output folders are supported.
Use `EXTRA_DATASETS` for recordings written with a custom `--dataset-dir`.

The existing smoke runs test logging and recording; their short horizons do not establish policy quality.


In [ ]:
RUN_NAMES = []  # e.g. ["p2_recording_verified_enabled", "metrics_monitoring_smoke_eval_verified"]
EXTRA_DATASETS = {}  # e.g. {"my_run": [ROOT / "datasets" / "my_recording"]}
SMOOTHING_POINTS = 1  # Reporting barriers, weighted by completed-race count; 1 = no smoothing.
EXPORT = False
EXPORT_DIR = OUTPUT_ROOT / "analysis" / "run_review"

if RUN_NAMES:
    selected_paths = [OUTPUT_ROOT / name for name in RUN_NAMES]
    missing = [str(p) for p in selected_paths if p.resolve() not in run_paths]
    if missing:
        raise ValueError(f"No saved run artifacts found: {missing}")
else:
    candidates = [p for p in run_paths if (p / "race_metrics.jsonl").exists() or (p / "team_metrics.jsonl").exists()]
    selected_paths = sorted(candidates, key=lambda p: max(f.stat().st_mtime for f in (p / "race_metrics.jsonl", p / "team_metrics.jsonl") if f.exists()))[-1:]
    if not selected_paths:
        selected_paths = run_paths[-1:]
    print("Preview selection:", [str(p.relative_to(OUTPUT_ROOT)) for p in selected_paths])

runs = [load_run(p, label=str(p.relative_to(OUTPUT_ROOT)),
                 dataset_dirs=EXTRA_DATASETS.get(str(p.relative_to(OUTPUT_ROOT)), ())) for p in selected_paths]
metadata = pd.DataFrame([r.metadata for r in runs])
races, agents, updates, evaluations, clips = [combine(runs, table) for table in
                                            ("races", "agents", "updates", "evaluations", "clips")]
team_races = combine(runs, "team_races")
recording_windows = combine(runs, "recording_windows")
display(metadata)
display(evaluations)
if races.empty and team_races.empty:
    print("No P0 race records in this selection. Update diagnostics may still be available.")
figures = {}


## Training and checkpoint-selection curves
Training plots use completed episodes and the aggregate environment-step count at the reporting barrier,
not an invented exact global completion time. Continuous and finite training get separate figures.
Checkpoint-selection evaluation gets separate curves, grouped by recorded evaluation conditions.
Missing losses or diagnostics are displayed as unavailable.

Self-play gets separate symmetric team training curves and a per-team table. Its completion/progress win is not first place; incomplete budget-cut episodes are excluded from completed-race curves. Self-play evaluation has its own per-team/per-map table and curves with race counts, seeds, checkpoint hashes, and protocol identities. Selection, final, and custom protocols stay separate; checkpoints are never pooled. The original fixed-opponent evaluation/seed-comparison tables below remain separate from self-play.

In [ ]:
figures.update(learning_curves(races, window=SMOOTHING_POINTS))
figures.update(optimizer_curves(updates))
figures.update(evaluation_curves(races))
figures.update(selfplay_curves(team_races, updates))
selfplay_evaluation = summarize_selfplay_evaluations(team_races)
figures.update(selfplay_evaluation_curves(selfplay_evaluation))
display(team_races)
display(selfplay_evaluation)
plt.show()


## Evaluation outcomes and finish times
The comparison uses the **latest checkpoint-selection evaluation** and any selected **standalone report**.
Selection, final, and custom protocols remain separate. Tables retain checkpoint identity, counts,
and a fingerprint of recorded seeds, map assignments, horizon, and physics conditions.
Compare provenance too: matching fingerprints do not prove every opponent or unrecorded setting matches.

Finish times include only clean learner finishers; plots show completion counts and sample sizes.
The per-car table includes learners and opponents. Training summaries describe all completed logged
training episodes, rather than just recorded clips, and are kept in a separate table.


In [ ]:
training = races.loc[races.phase == "training"] if not races.empty else races
chosen_eval = latest_evaluations(races)
training_summary = summarize_races(training)
evaluation_summary = summarize_races(chosen_eval)
overall_evaluation = summarize_races(chosen_eval, by_map=False)
if not chosen_eval.empty:
    selected_ids = chosen_eval[["run_key", "evaluation_id"]].drop_duplicates()
    eval_agents = agents.merge(selected_ids, on=["run_key", "evaluation_id"], validate="many_to_one")
else:
    eval_agents = pd.DataFrame()
per_car = summarize_agents(eval_agents)
display(evaluation_summary)
display(overall_evaluation)
display(per_car)
display(training_summary)
figures.update(outcome_plots(evaluation_summary))
figures.update(finish_time_plots(eval_agents))
plt.show()


## Variation across training seeds
Assign each selected evaluation run to an experiment arm below. Choose one run/checkpoint per training
seed in each arm and protocol. Repeated runs of the same training seed are not independent seeds.
Different recorded evaluation conditions stay in separate groups.

Each seed contributes one run-level value, regardless of evaluation episode count. Error bars are
**sample standard deviation across training seeds**, not episode-level uncertainty or confidence
intervals. With one measured seed, SD is missing. Inspect the outcome tables for race denominators.


In [ ]:
COMPARISON_GROUPS = {}  # Folder label -> arm, e.g. {"base_seed1/eval": "base", "base_seed2/eval": "base"}
unknown = set(COMPARISON_GROUPS) - {r.metadata["run"] for r in runs}
if unknown:
    raise ValueError(f"Comparison groups refer to unselected runs: {sorted(unknown)}")
groups = {r.metadata["run_key"]: COMPARISON_GROUPS[r.metadata["run"]]
          for r in runs if r.metadata["run"] in COMPARISON_GROUPS}
seed_summary = aggregate_training_seeds(evaluation_summary, groups)
display(seed_summary)
figures.update(seed_variation_plots(seed_summary))
plt.show()


## Find recorded races and event clips
This index reads clip metadata only. Representative samples and event-selected clips remain visibly
separate: event clips cannot estimate behavior frequencies. Partial/open clips remain identifiable.
Outcome filters use reported training/evaluation race facts when available; missing outcomes do not count as failures.

Choose a row and copy the generated replay command into a terminal with a graphical display.
Review uses the recording's policy-version interval; a corresponding saved checkpoint may not exist.
Use the reviewer below for synchronized playback and annotation editing. Exact checkpoint filtering matches only recorded checkpoint paths/hashes; use policy-version intervals for training clips without an exact saved checkpoint identity.

PPO and fixed-opponent MAPPO evaluation clips retain the exact checkpoint path/hash and protocol. Selection evaluations do not compute rewards, so those channels remain unavailable; standalone evaluations retain their computed reward components. A PPO selection evaluator can stop when its learner exits while opponents are still active; that recording is marked `evaluator_boundary` and incomplete.


In [ ]:
CLIP_FILTERS = dict(run=None, map_id=None, kind=None, event=None, agent=None, outcome=None, checkpoint=None, policy_version=None, team=None, phase=None, protocol=None, window_index=None)
# kind: "representative_race" / "representative_segment" / "event_clip"; event: "candidate_pass", "terminal", etc.
# outcome: "both_finished", "first_place", "sweep", "any_learner_collision_dnf".
# Self-play: set team="team_a" or "team_b"; outcomes: "both_finished", "any_learner_collision_dnf", "win".
filtered_clips = filter_clips(clips, **CLIP_FILTERS).reset_index(drop=True)
columns = [c for c in ["run", "clip_id", "map_id", "phase", "protocol", "evaluation_id", "checkpoint", "kind", "retention_reasons", "complete",
    "status", "end_reason", "policy_version_start", "policy_version_end", "team_policy_versions_start", "team_policy_versions_end", "agent_teams", "episode_id",
    "recording_window_index", "recording_progress_start", "recording_progress_clock", "coverage_scope",
    "start_physics_index", "end_physics_index", "both_finished", "any_learner_collision_dnf"]
    if c in filtered_clips]
display(recording_windows)  # Reserved caps and actual usage, including unused later windows.
display(filtered_clips[columns])
CLIP_ROW = 0
if not filtered_clips.empty:
    print(replay_command(filtered_clips.iloc[CLIP_ROW], ROOT, speed=1))
else:
    print("No matching clips. Record with --record-races, or set EXTRA_DATASETS for a custom location.")


## Synchronized clip review and annotations

Choose a clip and load an interval (at most 2,000 physics frames by default). Long races initially select
only their first window; change the start/end controls to inspect another interval. The clip index stays
in memory, while only overlapping trajectory chunks are read. Frame indices in the loader are inclusive;
annotation boundaries use **[start, end)**. Boundary 0 of the player shows the first pre-state, then each
physics interval's post-state. Commands/rewards describe their labeled interval, not the next decision.

Use **Play/Pause**, the boundary slider, speed, loop, or the event picker. Car colors are consistent;
the original learner/opponent view uses blue/orange; self-play labels the explicit teams. The map uses local map assets. Observed speed/steering,
applied drive references, recorded reference rates (wheel acceleration in rad/s² for wheel control),
relative gaps, reward components, and measured/heuristic events share recorded simulation time.
Unavailable values remain blank. Kernel/render latency may slow playback.

For an **individual** segment, choose one actor and any target cars. Set start/end at the cursor,
choose or type a tactic label, and save. Then choose **New segment** for another maneuver.
For a **combined** segment, select multiple actors and at least two saved individual segments from
the same race. Its interval must contain those segments. Record participant roles/results, role changes
at the cursor, and the combined outcome. Order/overlap is derived from the linked intervals.
Team benefit and inferred coordination require separate evidence; neither follows from the tactic label.

The **Saved** picker reopens segments for editing; save preserves their IDs. Segments from another
clip/window identify the source interval to reopen. Overlapping and user-created labels are allowed.
Annotations default to `RUN_FOLDER/review/annotations.json`, outside raw data. Concurrent edits are
rejected until you explicitly reload the saved file. No annotations are written until you press Save.

For self-play clips, `team="team_a"` (or another team name) can be combined with `outcome="both_finished"`, `"any_learner_collision_dnf"`, or `"win"` in `CLIP_FILTERS`. Use `phase="evaluation"` and `protocol="final"` (or `"selection"`/`"custom"`) to filter recorded evaluations. Their saved pair path or hash can be used as `checkpoint`. Removed cars disappear after clearance; their lifecycle facts stay recorded.

Windowed recordings expose `window_index` in `CLIP_FILTERS` and a reserved/used budget table. `representative_segment` means sampling resumed during a race; it is never labeled a complete race. Window boundaries discard pre-event history, so clips do not bridge unrecorded gaps. Parallel progress is the last completed collection barrier, not an exact per-frame global timestamp.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
from src.analysis.clip_review import load_window
from src.analysis.review_widget import ClipReviewer

MAX_REVIEW_FRAMES = 2000
ANNOTATION_PATH = None  # Optional shared file, outside raw recording directories.
if "reviewer" in globals() and reviewer is not None:
    reviewer.close()
reviewer = None
review_output = widgets.Output()

if filtered_clips.empty:
    print("Select a run with P2 recordings to enable clip review.")
else:
    clip_picker = widgets.Dropdown(options=[
        (f"{row['run']} · {row['kind']} · {row['clip_id']}", i)
        for i, row in filtered_clips.iterrows()], description="Clip",
        layout=widgets.Layout(width="95%"))
    review_start = widgets.IntText(description="First frame")
    review_end = widgets.Text(description="Last frame", placeholder="Blank = retained end")
    load_button = widgets.Button(description="Load interval", button_style="primary")

    def choose_interval(change=None):
        row = filtered_clips.iloc[clip_picker.value]
        start = int(row["start_physics_index"])
        stop = row.get("end_physics_index")
        review_start.value = start
        review_end.value = str(min(int(stop), start + MAX_REVIEW_FRAMES - 1)) if pd.notna(stop) else ""

    def open_review(_=None):
        global reviewer
        with review_output:
            clear_output(wait=True)
            if reviewer is not None:
                reviewer.close()
                reviewer = None
            row = filtered_clips.iloc[clip_picker.value].to_dict()
            try:
                window = load_window(row["dataset_dir"], row, start=review_start.value,
                    end=int(review_end.value) if review_end.value.strip() else None,
                    max_frames=MAX_REVIEW_FRAMES)
                annotation_path = ANNOTATION_PATH or Path(row["run_key"]) / "review" / "annotations.json"
                reviewer = ClipReviewer(window, annotation_path=annotation_path, maps_dir=ROOT / "maps")
                display(reviewer.widget)
            except (ValueError, OSError) as error:
                print(f"Cannot load interval: {error}")

    clip_picker.observe(choose_interval, names="value")
    load_button.on_click(open_review)
    choose_interval()
    display(widgets.VBox([clip_picker, widgets.HBox([review_start, review_end, load_button]), review_output]))
    open_review()


## Export analysis
Set `EXPORT = True` in the selection cell to write CSV tables and standalone PNG/PDF figures.
The manifest records selected folders, configuration/checkpoint hashes, evaluation conditions, smoothing,
and seed-group assignments. Exports use the files as read during this notebook execution;
rerun after training writes new data. Raw logs and recordings remain the source of truth.


In [ ]:
if EXPORT:
    import json
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {"runs": metadata, "evaluations": evaluations, "training_summary": training_summary,
              "evaluation_by_map": evaluation_summary, "evaluation_overall": overall_evaluation,
              "per_car": per_car, "training_seed_summary": seed_summary, "clips": filtered_clips, "selfplay_team_races": team_races, "selfplay_evaluation_by_map": selfplay_evaluation, "recording_windows": recording_windows}
    for name, table in tables.items():
        table.to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    export_figures(figures, EXPORT_DIR / "figures")
    manifest = {"runs": [r.metadata for r in runs], "smoothing_reporting_points": SMOOTHING_POINTS,
                "comparison_groups": COMPARISON_GROUPS, "clip_filters": CLIP_FILTERS,
                "evaluation_snapshots": json.loads(evaluations.to_json(orient="records"))}
    (EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
    print(f"Exported {len(figures)} figures and {len(tables)} tables to {EXPORT_DIR}")
